# Kontrola opublikowanego raportu z 26.10.2026

Notatnik odtwarza wskaźniki audytu z minimalnego wyciągu `dowody.json`. Wyciąg pochodzi z aktualnego archiwum wydania; zawiera identyfikatory obserwacji, status każdej strony, źródła przykładowych zdarzeń i wyniki próby przeglądarkowej. Skróty SHA-256 identyfikują pełne pliki źródłowe. Nie jest to test nowego wdrożenia.

Zależności: wyłącznie Python 3.9+. Uruchomić z katalogu notatnika albo repozytorium. Pełne źródło: https://github.com/MateuszD0/Pracownie/releases/tag/monitor-report-2026-10-26 .

In [1]:
import json
from collections import Counter
from datetime import datetime
from pathlib import Path
from zoneinfo import ZoneInfo

candidates = [Path('dowody.json'), Path('docs/audyt-2026-10-26/dowody.json')]
source = next(p for p in candidates if p.is_file())
data = json.loads(source.read_text())
print(data['source']['run_id'])
print(data['source']['release_url'])

2026-10-26T151141.729914-0000-cfafb6fe
https://github.com/MateuszD0/Pracownie/releases/tag/monitor-report-2026-10-26


## Kompletność — trzy odrębne definicje

Świeży odczyt mówi tylko o obecności zapisu dzisiaj. Porównywalna strona wymaga pełnej podstawy z obu kolejnych dni. Pracownia z kompletem ma porównywalną stronę główną i stałą kartę. Nie zakładamy kompletności wszystkich interakcji.

In [2]:
firms = data['pages_by_firm']
pages = [p for firm in firms for p in firm['pages']]
fresh = sum(bool(p['current_read']) for p in pages)
partial_fresh = sum(bool(p['current_read']) and p['data_quality'] == 'partial' for p in pages)
paired = sum(p['comparable_daily'] for p in pages)
complete_firms = sum(len(f['pages']) == 2 and all(p['comparable_daily'] for p in f['pages']) for f in firms)
assert (len(firms), len(pages), fresh, partial_fresh, paired, complete_firms) == (66, 132, 128, 4, 121, 60)
print(f'Pracownie z obiema stronami porównanymi: {complete_firms}/{len(firms)}')
print(f'Porównywalne strony: {paired}/{len(pages)}')
print(f'Świeże odczyty: {fresh}/{len(pages)}, w tym {partial_fresh} częściowe')
for f in firms:
    gaps = [p for p in f['pages'] if not p['comparable_daily']]
    if gaps:
        print(f['domain'], ' | '.join(p['page'] + ': ' + (p['error_text'] or p['comparison_label'] or 'brak pełnej pary') for p in gaps))

Pracownie z obiema stronami porównanymi: 60/66
Porównywalne strony: 121/132
Świeże odczyty: 128/132, w tym 4 częściowe
archido.pl glowna: Witryna blokuje dostęp ze skanera (403). | projekt: Nie ustalono poprawnej karty projektu.
dompp.pl glowna: Pierwszy zapis · brak porównywalnej bazy | projekt: Pierwszy zapis · brak porównywalnej bazy
hbstudio.pl projekt: Luka 2 dni · brak porównania z wczoraj
housecollection.pl glowna: Pierwszy zapis · brak porównywalnej bazy | projekt: Pierwszy zapis · brak porównywalnej bazy
lk-projekt.pl glowna: Witryna blokuje dostęp ze skanera (403). | projekt: Nie ustalono poprawnej karty projektu.
mgprojekt.com.pl glowna: Pierwszy zapis · brak porównywalnej bazy | projekt: Pierwszy zapis · brak porównywalnej bazy


## Obserwacje a ważność biznesowa

130 obserwacji wyglądu nie oznacza 130 błędów ani 130 zmian strategii. To miara struktury wewnętrznej listy. Główna tabela grupuje je i pokazuje tylko część tematów.

In [3]:
events = data['management_observations']
counts = Counter(e['business_bucket'] for e in events)
print(dict(counts))
print(f"Wygląd: {counts['design']}/{len(events)} = {counts['design']/len(events):.1%}")
assert len(events) == 177 and counts['design'] == 130
groups = data['digest_groups']
print('Tematy główne:', sum(g['overview'] for g in groups))
print('Tematy w pozostałych różnicach:', sum(not g['overview'] for g in groups))
for e in events:
    if e['domain'] == 'planujdom.pl' and e['business_bucket'] == 'promotions':
        assert e['commercial_status'] == 'not_checked' and not e['overview_eligible']
        print(e['domain'], e['page'], 'w głównej tabeli:', e['overview_eligible'])
        print(e['sentence'])

{'promotions': 4, 'functions': 2, 'copy': 41, 'design': 130}
Wygląd: 130/177 = 73.4%
Tematy główne: 11
Tematy w pozostałych różnicach: 11
planujdom.pl glowna w głównej tabeli: False
Zniknął komunikat: „WYBRANE PROJEKTY Z RABATEM -15% Łap okazję!!! Do końca promocji: [licznik] Zobacz >”. Weryfikacja oferty w serwisie nie została wykonana dla tego zapisu.
planujdom.pl projekt w głównej tabeli: False
Zniknął komunikat: „WYBRANE PROJEKTY Z RABATEM -15% Łap okazję!!! Do końca promocji: [licznik] Zobacz >”. Weryfikacja oferty w serwisie nie została wykonana dla tego zapisu.


## Dostępność raportu i wysyłka

Czas jest przeliczony z UTC na Europe/Warsaw. Wynik SMTP nie oznacza jeszcze doręczenia do skrzynki; w badanym przebiegu wysyłkę w ogóle pominięto.

In [4]:
published = datetime.fromisoformat(data['source']['first_published_at'].replace('Z', '+00:00')).astimezone(ZoneInfo('Europe/Warsaw'))
deadline = published.replace(hour=8, minute=0, second=0, microsecond=0)
print('Pierwsza publikacja:', published.isoformat())
print('Sekundy po 08:00:', int((published-deadline).total_seconds()))
assert int((published-deadline).total_seconds()) == 1606
job = data['email_jobs'][0]
send = next(s for s in job['steps'] if s['name'] == 'Wyślij porównanie')
print('Workflow:', job['conclusion'], '| krok wysyłki:', send['conclusion'])
assert job['conclusion'] == 'success' and send['conclusion'] == 'skipped' 

Pierwsza publikacja: 2026-10-26T08:26:46+01:00
Sekundy po 08:00: 1606
Workflow: success | krok wysyłki: skipped


## Próba opublikowanego HTML i rozmiar archiwum

Wpisano `mgprojekt.com.pl` w wyszukiwarkę przenośnego HTML. Wynik się nie zawęził. Link do szczegółów otworzył właściwą sekcję. Rozmiary pochodzą z API GitHuba; projekcja 180 dni to scenariusz przy stałym rozmiarze, nie zmierzona historia.

In [5]:
checks = data['flow_checks']
print(json.dumps(checks, ensure_ascii=False, indent=2))
assert checks['search_before_visible_domains'] == checks['search_after_visible_domains'] == 66
assert checks['scripts'] == 0 and checks['comparison_opened_details']
for asset in data['asset_sizes']:
    print(asset['name'].rsplit('-',1)[-1], round(asset['size']/1_000_000, 2), 'MB')
archive = next(a for a in data['asset_sizes'] if a['name'].endswith('cfafb6fe.zip'))
print('180 × obecne pełne archiwum:', round(180*archive['size']/1_000_000_000, 1), 'GB')

{
  "search_before_visible_domains": 66,
  "search_after_visible_domains": 66,
  "scripts": 0,
  "comparison_href": "#firma-zmiana-1b99d38369616d",
  "comparison_opened_details": true,
  "mobile_body_overflow": false,
  "mobile_table_overflows": 5
}
porownanie.html 6.34 MB
widok.zip 31.39 MB
cfafb6fe.zip 263.55 MB
180 × obecne pełne archiwum: 47.4 GB


## Granice i reprodukcja

Nie wnioskować o zakończeniu promocji Planujdom z samego braku baneru. Nie wyciągać trendów terminowości z jednego dnia. Nie uznawać wszystkich obserwacji wyglądu za nieistotne.

Wyciąg utworzono z `oferty.json` pobranego z pełnego ZIP wydania, po ponownym przeliczeniu `build_management(view)` i sprawdzeniu zgodności ze wskaźnikami zapisanymi w publikacji. Wynik przeglądarkowy dotyczy pobranego HTML; brakujące style GitHuba nie były podstawą oceny jego wyglądu. Szczegółowy zakres i plan: `../AUDYT_I_PLAN_DASHBOARDU_2026-10-26.md`.